# 🧵 AI Queue Engine

DB-backed queue utilities for multi-instance-safe AI job processing.

This notebook adds:
- Queue model (`AIJob`)
- Lease-based dequeue (`acquire_next_job`)
- Strict state transitions
- Retry + cancellation semantics

In [ ]:
#| default_exp utils_ai_queue

In [ ]:
#| export
from datetime import datetime, timedelta
from typing import Optional, Callable, Any, Dict, List
import json
import traceback
import logging

from sqlalchemy import text
from fastsql import Database
from fh_saas.db_host import gen_id, timestamp

logger = logging.getLogger(__name__)

In [ ]:
#| export
class AIJob:
    id: str
    tenant_id: str
    job_type: str
    status: str  # queued, processing, streaming, done, failed, canceled
    payload_json: str
    result_json: str = None
    error_log: str = None
    progress: int = 0
    cancel_requested: bool = False
    lease_until: str = None
    worker_id: str = None
    attempt: int = 0
    max_retries: int = 2
    created_at: str = None
    updated_at: str = None
    started_at: str = None
    completed_at: str = None

---
## Queue Manager

In [ ]:
#| export
class AIQueueManager:
    '''DB-backed queue manager with lease semantics for multi-instance workers.'''

    def __init__(self, db: Database):
        self.db = db
        self.ai_jobs = db.create(AIJob, name='ai_jobs', pk='id')

    def enqueue(self, *, tenant_id: str, job_type: str, payload: Dict[str, Any], max_retries: int = 2) -> str:
        job_id = gen_id()
        now = timestamp()
        self.ai_jobs.insert(AIJob(
            id=job_id,
            tenant_id=tenant_id,
            job_type=job_type,
            status='queued',
            payload_json=json.dumps(payload),
            max_retries=max_retries,
            created_at=now,
            updated_at=now,
        ))
        self.db.conn.commit()
        return job_id

    def get_job(self, job_id: str) -> Optional[AIJob]:
        try:
            return self.ai_jobs[job_id]
        except Exception:
            return None

    def list_jobs(self, tenant_id: Optional[str] = None, status: Optional[str] = None, limit: int = 100) -> List[AIJob]:
        where = []
        where_args: Dict[str, Any] = {}
        if tenant_id:
            where.append('tenant_id = :tenant_id')
            where_args['tenant_id'] = tenant_id
        if status:
            where.append('status = :status')
            where_args['status'] = status
        where_sql = ' AND '.join(where) if where else None
        return self.ai_jobs(where=where_sql, where_args=where_args or None, order_by='created_at ASC', limit=limit)

    def request_cancel(self, job_id: str) -> bool:
        job = self.get_job(job_id)
        if job is None:
            return False
        if job.status in ('done', 'failed', 'canceled'):
            return False
        self.ai_jobs.update(id=job_id, cancel_requested=True, updated_at=timestamp())
        self.db.conn.commit()
        return True

    def acquire_next_job(self, *, worker_id: str, lease_seconds: int = 30) -> Optional[AIJob]:
        '''Lease one queued job atomically-like using conditional update loop.'''
        now = datetime.utcnow().isoformat()
        candidates = self.ai_jobs(
            where="status = :status AND (lease_until IS NULL OR lease_until < :now)",
            where_args={'status': 'queued', 'now': now},
            order_by='created_at ASC',
            limit=10,
        )
        if not candidates:
            return None

        lease_until = (datetime.utcnow() + timedelta(seconds=lease_seconds)).isoformat()

        for cand in candidates:
            sql = text(
                '''
                UPDATE ai_jobs
                SET status = :processing,
                    worker_id = :worker_id,
                    lease_until = :lease_until,
                    started_at = COALESCE(started_at, :now),
                    updated_at = :now
                WHERE id = :id
                  AND status = :queued
                  AND (lease_until IS NULL OR lease_until < :now)
                '''
            )
            params = {
                'processing': 'processing',
                'worker_id': worker_id,
                'lease_until': lease_until,
                'now': now,
                'id': cand.id,
                'queued': 'queued',
            }
            res = self.db.conn.execute(sql, params)
            if res.rowcount and res.rowcount > 0:
                self.db.conn.commit()
                return self.ai_jobs[cand.id]

        self.db.conn.commit()
        return None

    def heartbeat(self, job_id: str, *, worker_id: str, lease_seconds: int = 30) -> bool:
        job = self.get_job(job_id)
        if not job or job.worker_id != worker_id or job.status not in ('processing', 'streaming'):
            return False
        lease_until = (datetime.utcnow() + timedelta(seconds=lease_seconds)).isoformat()
        self.ai_jobs.update(id=job_id, lease_until=lease_until, updated_at=timestamp())
        self.db.conn.commit()
        return True

    def complete(self, job_id: str, *, worker_id: str, result: Optional[Dict[str, Any]] = None) -> bool:
        job = self.get_job(job_id)
        if not job or job.worker_id != worker_id:
            return False
        self.ai_jobs.update(
            id=job_id,
            status='done',
            result_json=json.dumps(result) if result is not None else None,
            progress=100,
            lease_until=None,
            completed_at=timestamp(),
            updated_at=timestamp(),
        )
        self.db.conn.commit()
        return True

    def fail(self, job_id: str, *, worker_id: str, error: Exception) -> bool:
        job = self.get_job(job_id)
        if not job or job.worker_id != worker_id:
            return False

        error_msg = f"{type(error).__name__}: {str(error)}\n{traceback.format_exc()}"
        next_attempt = int(job.attempt or 0) + 1

        if next_attempt < int(job.max_retries or 1):
            self.ai_jobs.update(
                id=job_id,
                status='queued',
                attempt=next_attempt,
                error_log=error_msg,
                worker_id=None,
                lease_until=None,
                updated_at=timestamp(),
            )
        else:
            self.ai_jobs.update(
                id=job_id,
                status='failed',
                attempt=next_attempt,
                error_log=error_msg,
                worker_id=None,
                lease_until=None,
                completed_at=timestamp(),
                updated_at=timestamp(),
            )

        self.db.conn.commit()
        return True

    def execute_once(self, *, worker_id: str, task_func: Callable[..., Any], lease_seconds: int = 30) -> Optional[str]:
        '''Acquire and execute one job for this worker.'''
        job = self.acquire_next_job(worker_id=worker_id, lease_seconds=lease_seconds)
        if not job:
            return None

        if job.cancel_requested:
            self.ai_jobs.update(
                id=job.id,
                status='canceled',
                worker_id=None,
                lease_until=None,
                completed_at=timestamp(),
                updated_at=timestamp(),
            )
            self.db.conn.commit()
            return job.id

        payload = json.loads(job.payload_json or '{}')

        try:
            result = task_func(**payload)
            if hasattr(result, '__iter__') and not isinstance(result, (dict, list, str, bytes)):
                chunks = list(result)
                self.complete(job.id, worker_id=worker_id, result={'chunks': chunks})
            else:
                self.complete(job.id, worker_id=worker_id, result=result if isinstance(result, dict) else {'value': result})
        except Exception as e:
            logger.error(f'AI job {job.id} failed: {e}', exc_info=True)
            self.fail(job.id, worker_id=worker_id, error=e)

        return job.id